# JP Morgan Chase — AI Finance Agent
## Lab 06 : LangChain Agent — Advanced Workflows
**Author :** Fabrice William FOMHOM  
**Date :** March 2026  
**Objective :** Build a multi-step reasoning agent using LangChain
that can use multiple tools, remember conversation history,
and perform complex financial analysis autonomously

## What Makes LangChain Different
- **Memory** : Agent remembers previous questions
- **Tools** : Agent chooses which tool to use
- **Reasoning** : Agent thinks step by step (ReAct pattern)
- **Autonomy** : Agent decides HOW to answer, not just what

In [1]:
# ============================================================
# JP Morgan Chase — Lab 06 : LangChain Multi-Agent System
# Step 1 : Import LangChain libraries
# ============================================================

import os
import pandas as pd
import numpy as np
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# API Key
os.environ["ANTHROPIC_API_KEY"] = "YOUR_API_KEY_HERE"

# Initialize Claude via LangChain
llm = ChatAnthropic(
    model       = "claude-sonnet-4-20250514",
    max_tokens  = 2048,
    temperature = 0
)

# Load dataset
BASE_DIR = r"C:\Users\HP\jpmorgan_finance_agent"
DATA_DIR = os.path.join(BASE_DIR, "data")

df = pd.read_csv(
    os.path.join(DATA_DIR, "jpmorgan_transactions.csv"),
    parse_dates=["date"]
)

print("✅ LangChain imported successfully")
print("✅ Claude LLM initialized via LangChain")
print(f"✅ Dataset loaded : {df.shape[0]} rows")
print(f"✅ LangGraph version ready for multi-agent system")

✅ LangChain imported successfully
✅ Claude LLM initialized via LangChain
✅ Dataset loaded : 1000 rows
✅ LangGraph version ready for multi-agent system


In [5]:
# ============================================================
# Step 2 : Define Data Analyst Tools
# These are the functions our analyst agent can use
# ============================================================

@tool
def get_portfolio_summary() -> str:
    """Get overall portfolio statistics for JP Morgan transactions."""
    return f"""
    PORTFOLIO SUMMARY:
    - Total transactions  : {len(df):,}
    - Total volume        : ${df['amount'].sum():,.2f}
    - Average transaction : ${df['amount'].mean():,.2f}
    - Date range          : {df['date'].min().date()} to {df['date'].max().date()}
    """

@tool
def get_fraud_analysis() -> str:
    """Get detailed fraud analysis of the transaction portfolio."""
    fraud_df = df[df['is_fraud'] == 1]
    normal_df = df[df['is_fraud'] == 0]
    return f"""
    FRAUD ANALYSIS:
    - Total fraud cases   : {len(fraud_df)}
    - Fraud rate          : {df['is_fraud'].mean()*100:.2f}%
    - Total fraud losses  : ${fraud_df['amount'].sum():,.2f}
    - Avg fraud amount    : ${fraud_df['amount'].mean():,.2f}
    - Avg normal amount   : ${normal_df['amount'].mean():,.2f}
    - Fraud is {fraud_df['amount'].mean()/normal_df['amount'].mean():.1f}x larger than normal
    """

@tool
def get_category_risk() -> str:
    """Get fraud risk breakdown by merchant category."""
    cat = df.groupby('category').agg(
        total       = ('is_fraud', 'count'),
        fraud_cases = ('is_fraud', 'sum')
    )
    cat['fraud_rate'] = (cat['fraud_cases']/cat['total']*100).round(2)
    cat = cat.sort_values('fraud_rate', ascending=False)
    return f"CATEGORY RISK REPORT:\n{cat.to_string()}"

@tool
def get_monthly_trend() -> str:
    """Get monthly transaction and fraud trends."""
    monthly = df.groupby('month_name').agg(
        transactions = ('transaction_id', 'count'),
        volume       = ('amount', 'sum'),
        fraud_cases  = ('is_fraud', 'sum')
    ).reindex(['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec'])
    return f"MONTHLY TREND:\n{monthly.to_string()}"

@tool
def flag_suspicious_transaction(amount: float, category: str) -> str:
    """
    Flag a transaction as suspicious based on amount and category.
    
    Parameters:
        amount   : transaction amount in dollars
        category : merchant category of the transaction
    """
    avg_fraud  = df[df['is_fraud']==1]['amount'].mean()
    cat_risk   = df.groupby('category')['is_fraud'].mean()
    risk_score = 0

    if amount > avg_fraud:
        risk_score += 40
    if amount > 500:
        risk_score += 20
    if category in cat_risk.index:
        if cat_risk[category] > 0.05:
            risk_score += 30
        elif cat_risk[category] > 0.03:
            risk_score += 15

    level = "🔴 HIGH" if risk_score>60 else "🟡 MEDIUM" if risk_score>30 else "🟢 LOW"

    return f"""
    TRANSACTION RISK ASSESSMENT:
    - Amount        : ${amount:,.2f}
    - Category      : {category}
    - Risk Score    : {risk_score}/100
    - Risk Level    : {level}
    - Avg fraud amt : ${avg_fraud:,.2f}
    - Recommendation: {'BLOCK and review immediately' if risk_score>60 else 'Monitor closely' if risk_score>30 else 'Approve — low risk'}
    """

# List all tools
tools = [
    get_portfolio_summary,
    get_fraud_analysis,
    get_category_risk,
    get_monthly_trend,
    flag_suspicious_transaction
]

print("✅ Data Analyst tools created :")
for t in tools:
    print(f"   → {t.name}")

✅ Data Analyst tools created :
   → get_portfolio_summary
   → get_fraud_analysis
   → get_category_risk
   → get_monthly_trend
   → flag_suspicious_transaction


In [6]:
# ============================================================
# Step 3 : Build the 3 JP Morgan Agents
# Each agent has a different role and personality
# ============================================================

memory = MemorySaver()

# ── AGENT 1 : Data Analyst ───────────────────────────────────
analyst_prompt = """You are Alex, a Junior Data Analyst at JP Morgan Chase.

YOUR ROLE:
- Run data queries using your tools
- Calculate statistics and identify patterns
- Report findings clearly with numbers
- Always use tools before answering

YOUR PERSONALITY:
- Detail-oriented and precise
- Always cite specific numbers
- Flag anything suspicious immediately
- End reports with: 'Analyst Alex — Analysis Complete'
"""

analyst_agent = create_react_agent(
    llm,
    tools   = tools,
    prompt  = analyst_prompt,
    checkpointer = memory
)

# ── AGENT 2 : Supervisor ─────────────────────────────────────
supervisor_prompt = """You are Sarah, a Senior Risk Supervisor at JP Morgan Chase.

YOUR ROLE:
- Review analyst findings for accuracy
- Identify business implications of the data
- Prioritize risks by severity
- Make tactical recommendations

YOUR PERSONALITY:
- Critical thinker who challenges assumptions
- Focuses on risk management
- Connects data findings to business impact
- End reports with: 'Supervisor Sarah — Review Complete'
"""

supervisor_agent = create_react_agent(
    llm,
    tools   = tools,
    prompt  = supervisor_prompt,
    checkpointer = memory
)

# ── AGENT 3 : Manager ────────────────────────────────────────
manager_prompt = """You are Michael, the VP of Risk Management at JP Morgan Chase.

YOUR ROLE:
- Ask strategic business questions
- Synthesize analyst and supervisor reports
- Make executive decisions
- Communicate findings to the board

YOUR PERSONALITY:
- Strategic and concise
- Focuses on bottom-line impact
- Speaks in business terms not technical ones
- End reports with: 'VP Michael — Decision Made'
"""

manager_agent = create_react_agent(
    llm,
    tools   = tools,
    prompt  = manager_prompt,
    checkpointer = memory
)

print("✅ JP Morgan Multi-Agent Team created :")
print("   → Alex    : Data Analyst")
print("   → Sarah   : Risk Supervisor")
print("   → Michael : VP of Risk Management")

✅ JP Morgan Multi-Agent Team created :
   → Alex    : Data Analyst
   → Sarah   : Risk Supervisor
   → Michael : VP of Risk Management


C:\Users\HP\AppData\Local\Temp\ipykernel_30136\678935173.py:24: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  analyst_agent = create_react_agent(
C:\Users\HP\AppData\Local\Temp\ipykernel_30136\678935173.py:47: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  supervisor_agent = create_react_agent(
C:\Users\HP\AppData\Local\Temp\ipykernel_30136\678935173.py:70: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  manager_agent = create_react_agent(


In [7]:
# ============================================================
# Step 4 : JP Morgan Multi-Agent Workflow
# Coordinates communication between all 3 agents
# ============================================================

def run_agent(agent, message, agent_name, thread_id="1"):
    """Run a single agent and return its response."""
    print(f"\n{'='*60}")
    print(f"🏦 {agent_name} is working...")
    print(f"{'='*60}")
    
    result = agent.invoke(
        {"messages": [HumanMessage(content=message)]},
        config={"configurable": {"thread_id": thread_id}}
    )
    
    response = result["messages"][-1].content
    print(response)
    return response


def jp_morgan_workflow(business_question):
    """
    Full JP Morgan Multi-Agent Workflow:
    Manager asks → Analyst researches → 
    Supervisor reviews → Manager decides
    """
    print(f"\n{'#'*60}")
    print(f"# JP MORGAN CHASE — MULTI-AGENT WORKFLOW")
    print(f"# Business Question: {business_question}")
    print(f"{'#'*60}")
    
    # ── Step 1 : Manager defines the task ────────────────────
    manager_task = run_agent(
        manager_agent,
        f"""As VP of Risk Management, you received this question: 
        '{business_question}'
        Define what analysis Alex the analyst should perform.""",
        "VP Michael (Manager)",
        thread_id="manager"
    )
    
    # ── Step 2 : Analyst performs the analysis ────────────────
    analyst_result = run_agent(
        analyst_agent,
        f"""The VP has requested this analysis: {manager_task}
        Use your tools to perform a complete analysis 
        and report your findings.""",
        "Alex (Data Analyst)",
        thread_id="analyst"
    )
    
    # ── Step 3 : Supervisor reviews ───────────────────────────
    supervisor_review = run_agent(
        supervisor_agent,
        f"""Review this analyst report: {analyst_result}
        Validate the findings, add risk implications, 
        and prepare recommendations for the VP.""",
        "Sarah (Supervisor)",
        thread_id="supervisor"
    )
    
    # ── Step 4 : Manager makes final decision ─────────────────
    final_decision = run_agent(
        manager_agent,
        f"""Supervisor Sarah's review: {supervisor_review}
        Make a final executive decision and summarize 
        in 3 bullet points for the board.""",
        "VP Michael (Final Decision)",
        thread_id="manager"
    )
    
    print(f"\n{'#'*60}")
    print(f"# WORKFLOW COMPLETE")
    print(f"{'#'*60}")
    
    return final_decision

print("✅ Multi-Agent Workflow ready!")
print("   Manager → Analyst → Supervisor → Manager")

✅ Multi-Agent Workflow ready!
   Manager → Analyst → Supervisor → Manager


In [8]:
# ============================================================
# Step 5 : Run the JP Morgan Multi-Agent Workflow
# Watch 3 agents collaborate to answer a business question
# ============================================================

result = jp_morgan_workflow(
    "What is our current fraud exposure and what immediate actions should we take to protect our customers?"
)



############################################################
# JP MORGAN CHASE — MULTI-AGENT WORKFLOW
# Business Question: What is our current fraud exposure and what immediate actions should we take to protect our customers?
############################################################

🏦 VP Michael (Manager) is working...
**Alex, based on your analysis, here's what I see:**

**CRITICAL FINDINGS:**
- We're hemorrhaging $23,737 in fraud losses with a 2.9% fraud rate
- Fraudulent transactions are 5.6x larger than legitimate ones ($819 vs $146)
- Restaurants are our highest risk category at 7.58% fraud rate
- July peak shows seasonal vulnerability with 4 fraud cases

**IMMEDIATE ACTIONS REQUIRED:**

1. **Restaurant Category Alert** - Implement enhanced monitoring for restaurant transactions over $400
2. **Summer Season Protocols** - Strengthen controls during July-August peak periods  
3. **Large Transaction Flags** - Any transaction 3x above category average needs real-time verification

In [9]:
# ============================================================
# Step 6 : Test different business scenarios
# ============================================================

# Scenario 1 : Real-time transaction decision
print("\n" + "🔴 "*20)
print("SCENARIO 1 : Real-time fraud alert")
print("🔴 "*20)

result2 = jp_morgan_workflow(
    "A $1,200 restaurant transaction just came in from customer CUST_0042. Should we approve or block it?"
)


🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 
SCENARIO 1 : Real-time fraud alert
🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 🔴 

############################################################
# JP MORGAN CHASE — MULTI-AGENT WORKFLOW
# Business Question: A $1,200 restaurant transaction just came in from customer CUST_0042. Should we approve or block it?
############################################################

🏦 VP Michael (Manager) is working...
**Alex, based on your analysis, here's the situation:**

**CRITICAL RISK INDICATORS:**
- **90/100 Risk Score** - This transaction is in our danger zone
- **$1,200 is 4x our average restaurant transaction** and 1.5x our typical fraud amount
- **Restaurant category has 7.58% fraud rate** - our highest risk merchant type
- **Amount exceeds our $300 enhanced monitoring threshold** by 400%

**IMMEDIATE DECISION REQUIRED:**

**BLOCK THIS TRANSACTION** - Risk is too high for automatic approval

**Alex, execute these steps immediately:**

1. **Customer Contact